# AI Explainer: SDXL Image Generator
This notebook generates AI images for your local YouTube Shorts pipeline.
**Make sure you are connected to a GPU runtime** (Runtime > Change runtime type > T4 GPU).

In [ ]:
# Step 1: Install Dependencies
!pip install diffusers transformers accelerate safetensors > /dev/null
print("✅ Dependencies installed!")

In [ ]:
# Step 2: Upload your manifest.json
from google.colab import files

print("Please upload the output/manifest.json from your local machine:")
uploaded = files.upload()

if 'manifest.json' in uploaded:
    print("✅ manifest.json uploaded successfully!")
else:
    print("❌ Error: Please make sure the file is named exactly 'manifest.json'")

In [ ]:
# Step 3: Generate Images and Zip them
import json
import torch
from diffusers import DiffusionPipeline
import os
import shutil

if not os.path.exists('manifest.json'):
    raise Exception("manifest.json not found! Please run the upload cell above.")

with open('manifest.json', 'r') as f:
    manifest = json.load(f)

print("🔄 Loading Stable Diffusion XL (this takes ~1 min)...")
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16"
)
pipe.to("cuda")
pipe.enable_model_cpu_offload()  # Keeps memory usage low on free Colab tier

os.makedirs("images", exist_ok=True)
generated_count = 0

print("\n🚀 Starting Generation...")
for seg in manifest.get("segments", []):
    if seg.get("visual_type") == "ai_image":
        seg_id = seg["id"]
        prompt = seg["visual_source"]
        out_file = f"images/seg_{seg_id:04d}.png"
        
        print(f"\n📸 Generating {out_file}...")
        print(f"Prompt: {prompt}")
        
        image = pipe(
            prompt=prompt,
            negative_prompt="low quality, blurry, distorted, deformed, ugly, bad anatomy, watermark, text, logo",
            num_inference_steps=25,
            width=1080,
            height=1920
        ).images[0]
        
        image.save(out_file)
        generated_count += 1

if generated_count > 0:
    print(f"\n✅ Successfully generated {generated_count} images! Zipping them up...")
    shutil.make_archive("generated_images", "zip", "images")
    print("\n⬇️ Download your 'generated_images.zip' from the folder icon on the left!")
else:
    print("\n✅ No AI images were required in this manifest! You're good to go.")